# FINANCIAL DATA ANALYSIS REPORT
## Based on Python for Finance Cookbook (Chapters 1 & 2)

## 0.Introduction
-   This notebook demonstrates financial data processing and analysis:
- Data collection (yfinance)
- Data cleaning
- Feature engineer
- Risk & return metrics
- Visualization
- Portfolio analysis

## 1.Import libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

## Download data (Crypto - BTC)

In [12]:
symbol = "BTC-USD"  # Bitcoin
start_date = "2020-01-01"
end_date = "2023-01-01"

df = yf.download(symbol, start=start_date, end=end_date, progress=False)

print("Raw Crypto Data (BTC):")
display(df.head())

Raw Crypto Data (BTC):


Price,Close,High,Low,Open,Volume
Ticker,BTC-USD,BTC-USD,BTC-USD,BTC-USD,BTC-USD
Date,,,,,
2020-01-01,7200.174316,7254.330566,7174.944336,7194.892090,18565664997
2020-01-02,6985.470215,7212.155273,6935.270020,7202.551270,20802083465
2020-01-03,7344.884277,7413.715332,6914.996094,6984.428711,28111481032
2020-01-04,7410.656738,7427.385742,7309.514160,7345.375488,18444271275
2020-01-05,7411.317383,7544.497070,7400.535645,7410.451660,19725074095


-   Các dữ liệu chứng khoán - Bitcoin đã được tải xuống thành công từ 01/01/2020 đến 01/01/2023 đầy đủ.

In [13]:
df = yf.download("BTC-USD",
                 period="30d",
                 interval="1h",
                 progress=False)
display(df.head())

Price,Close,High,Low,Open,Volume
Ticker,BTC-USD,BTC-USD,BTC-USD,BTC-USD,BTC-USD
Datetime,,,,,
2026-03-04 00:00:00+00:00,68185.593750,68397.906250,67934.507812,68317.765625,0
2026-03-04 01:00:00+00:00,68423.757812,68881.156250,68003.031250,68120.492188,1048698880
2026-03-04 02:00:00+00:00,68309.296875,68417.789062,68030.937500,68417.789062,0
2026-03-04 03:00:00+00:00,67669.062500,68326.679688,67449.273438,68316.031250,1025560576
2026-03-04 04:00:00+00:00,67811.960938,67911.585938,67496.234375,67778.382812,419692544


-   Vì giá trị Crypto như BTC liên tục biến động 24/7 thì khi ta quan sát theo mỗi ngày - daily sẽ mất rất nhiều thông tin quan trọng
-   Do đó ta sử dụng interval argument để chọn tần suất thể hiện , ở đây là 1 giờ 
-   Đây là tiền đề QUAN TRỌNG cho các bước xử lý tiếp theo như là Realized volatility, Intraday trading strategy, Volatility clustering, High-frequency analysis

In [8]:
btc = yf.Ticker("BTC-USD")
# Lấy dữ liệu theo giờ (intraday)
btc_intraday = btc.history(period="30d", interval="1h")

print("BTC Intraday Data:")
print(btc_intraday.head())

BTC Intraday Data:
                                   Open          High           Low         Close      Volume  Dividends  Stock Splits
Datetime                                                                                                              
2026-03-04 00:00:00+00:00  68317.765625  68397.906250  67934.507812  68185.593750           0        0.0           0.0
2026-03-04 01:00:00+00:00  68120.492188  68881.156250  68003.031250  68423.757812  1048698880        0.0           0.0
2026-03-04 02:00:00+00:00  68417.789062  68417.789062  68030.937500  68309.296875           0        0.0           0.0
2026-03-04 03:00:00+00:00  68316.031250  68326.679688  67449.273438  67669.062500  1025560576        0.0           0.0
2026-03-04 04:00:00+00:00  67778.382812  67911.585938  67496.234375  67811.960938   419692544        0.0           0.0


## 3.DATA CLEANING

In [ ]:
#Xóa bỏ các dữ liệu bị thiếu
df["Volume"] = df["Volume"].replace(0, np.nan)
df = df.dropna()

if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.droplevel(1)
    
df = df.dropna()

print("Cleaned Data:")
print(df.info())
display(df.head())

Cleaned Data:
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 358 entries, 2026-03-04 01:00:00+00:00 to 2026-04-02 16:00:00+00:00
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Close   358 non-null    float64
 1   High    358 non-null    float64
 2   Low     358 non-null    float64
 3   Open    358 non-null    float64
 4   Volume  358 non-null    float64
dtypes: float64(5)
memory usage: 16.8 KB
None


Price,Close,High,Low,Open,Volume
Datetime,,,,,
2026-03-04 01:00:00+00:00,68423.757812,68881.156250,68003.031250,68120.492188,1.048699e+09
2026-03-04 03:00:00+00:00,67669.062500,68326.679688,67449.273438,68316.031250,1.025561e+09
2026-03-04 04:00:00+00:00,67811.960938,67911.585938,67496.234375,67778.382812,4.196925e+08
2026-03-04 06:00:00+00:00,68486.242188,68650.125000,68099.585938,68209.109375,1.144762e+09
2026-03-04 07:00:00+00:00,69556.468750,69556.468750,68390.539062,68513.906250,8.318894e+08


-   Ở bước này ta sẽ xử lý dữ liệu bị thiếu như có vài row có Volumn = 0 
-   Nguyên nhân không phải do BTC có giao dịch mà do missing data hoặc lỗi từ yfinance
-   Thay vì giữ lại ta nên loại bỏ hẳn thay vì giữ lại sẽ gây khó hiểu cho model và nhiễu biến động

In [ ]:
from alpha_vantage.cryptocurrencies import CryptoCurrencies

crypto_api = CryptoCurrencies(key="YOUR_API_KEY", output_format="pandas")

df, meta = crypto_api.get_digital_currency_exchange_rate(
    from_currency="BTC",
    to_currency="USD"
)

display(df.transpose())

,Realtime Currency Exchange Rate
1. From_Currency Code,BTC
2. From_Currency Name,Bitcoin
3. To_Currency Code,USD
4. To_Currency Name,United States Dollar
5. Exchange Rate,67085.29000000
6. Last Refreshed,2026-04-02 17:03:04
7. Time Zone,UTC
8. Bid Price,67084.92100000
9. Ask Price,67085.32600000


-   Lấy tỷ giá thời gian thực của Bitcoin
-   Đoạn mã sử dụng API để truy xuất tỷ giá hối đoái theo thời gian thực giữa Bitcoin (BTC) và USD:
-   Phù hợp cho các ứng dụng cần dữ liệu live: theo dõi thị trường, hệ thống giao dịch tự động
-   Việc sử dụng API để lấy tỷ giá thời gian thực của Bitcoin là cần thiết trong các bài toán yêu cầu cập nhật dữ liệu liên tục. Tuy nhiên, phương pháp này không thể thay thế hoàn toàn các nguồn dữ liệu lịch sử như yfinance.

## 4. FEATURE ENGINEERING

In [ ]:
# Returns
df['Return'] = df['Close'].pct_change()
df['Log_Return'] = np.log(df['Close'] / df['Close'].shift(1))

# Rolling stats
df['MA20'] = df['Close'].rolling(20).mean()
df['STD20'] = df['Close'].rolling(20).std()

# Volatility (annualized)
df['Volatility'] = df['Log_Return'].rolling(20).std() * np.sqrt(252)

# Cumulative returns
df['Cumulative_Return'] = (1 + df['Return']).cumprod()

# Drawdown
rolling_max = df['Cumulative_Return'].cummax()
df['Drawdown'] = df['Cumulative_Return'] / rolling_max - 1

print("Feature Engineered Data:")
display(df.tail())


## 5. VISUALIZATION

In [ ]:
# Price + Moving Average
plt.figure()
plt.plot(df['Close'], label='Close')
plt.plot(df['MA20'], label='MA20')
plt.legend()
plt.title('Price & Moving Average')
plt.show()

# Returns
plt.figure()
plt.plot(df['Return'])
plt.title('Daily Returns')
plt.show()

# Volatility
plt.figure()
plt.plot(df['Volatility'])
plt.title('Rolling Volatility')
plt.show()

# Cumulative Return
plt.figure()
plt.plot(df['Cumulative_Return'])
plt.title('Cumulative Return')
plt.show()

# Drawdown
plt.figure()
plt.plot(df['Drawdown'])
plt.title('Drawdown')
plt.show()

# Distribution of returns
plt.figure()
plt.hist(df['Return'].dropna(), bins=50)
plt.title('Return Distribution')
plt.show()


## 6. RESAMPLING

In [ ]:
monthly_df = df.resample('M').agg({
    'Open': 'first',
    'High': 'max',
    'Low': 'min',
    'Close': 'last',
    'Volume': 'sum'
})

print("Monthly Data:")
display(monthly_df.head())


## 7. RISK METRICS

In [ ]:
mean_return = df['Return'].mean()
volatility = df['Return'].std()
sharpe_ratio = mean_return / volatility

max_drawdown = df['Drawdown'].min()

print("Risk Metrics:")
print(f"Mean Return: {mean_return:.6f}")
print(f"Volatility: {volatility:.6f}")
print(f"Sharpe Ratio: {sharpe_ratio:.6f}")
print(f"Max Drawdown: {max_drawdown:.6f}")

## 8. MULTI-ASSET ANALYSIS

In [ ]:
# Crypto portfolio
symbols = ['BTC-USD', 'ETH-USD', 'BNB-USD']
multi_df = yf.download(symbols, start=start_date, end=end_date, progress=False)['Close']

returns = multi_df.pct_change().dropna()

# Correlation
corr_matrix = returns.corr()
print("Crypto Correlation Matrix:")
display(corr_matrix)

# Heatmap
plt.figure()
plt.imshow(corr_matrix)
plt.colorbar()
plt.xticks(range(len(symbols)), symbols)
plt.yticks(range(len(symbols)), symbols)
plt.title('Crypto Correlation Heatmap')
plt.show()